> **2026-04-19 note — C value.** This notebook uses the current ship **ET-midnight snap convention** (correct per `CLAUDE.md`), but the fit run below sets `PHASE2_CONST = 2.0`. The ship value per `CLAUDE.md` is **C = 1.0** (empirical mean of phase-2 counts on the 5 h/m targets is 1.0; C=2 was an earlier ship-conservative midpoint from a wider 14h UTC-midnight window, now superseded). Headline numbers in this notebook's output reflect C=2. For the C=1 composition numbers that are ship-bound, see `findings/ridge_lambda_investigation.md` §4.3. **Do not copy `PHASE2_CONST = 2.0` into new code.**

# Proposed ship-stack test: Ridge (ET-midnight convention) + C=2 piecewise

**Intent:** evaluate the full proposed ship stack end-to-end on h/m targets.

**Convention (new, differs from prior tier-2 validation):**
- **Snap:** midnight ET on close−N days (4h later than our validated midnight-UTC snap).
- **Phase 1:** Ridge predicts reviews in `(midnight ET on close, midnight ET on close−N]` — N × 24h exactly.
- **Phase 2:** piecewise constant, **C = 2 reviews** over `(midnight ET on close, 10am ET)` — the final 10 hours.
- **Predicted total = Ridge_phase1 + 2**.
- **Actual total for target** = reviews in `(snap_time, close_ts]`. All h/m movies are Spring (EDT); scraper was on until right after 10am EST/EDT close for every h/m movie, so the actual count is exact.

**Setup:** refit Ridge on full 143-movie cohort under this ET convention (LOO per-snap, same 17 features as tier 2). Predict on h/m subset. Compare predicted_total vs actual_total per snap and per-movie.

Key metric: per-movie error (predicted_total − actual_total) at each snap. Large errors indicate the stack is unreliable for that target at that snap — important for strategy layer.


In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
if NB_DIR.name != 'notebooks':
    NB_DIR = NB_DIR / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import pickle
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold

import _helpers as H

print(f'cohort: {len(H.close_date_map)} movies')


## Noon-shift (same convention as prior validation — de-spike day-level reviews)


In [ ]:
_day_mask = H.reviews['timestamp_confidence'] == 'd'
_n_shifted = int(_day_mask.sum())
H.reviews.loc[_day_mask, 'estimated_timestamp'] = (
    H.reviews.loc[_day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)
H.first_review_ts = (
    H.reviews[H.reviews['movie_slug'].isin(H.close_date_map)]
    .groupby('movie_slug')['estimated_timestamp'].min()
)
_new_first = H.first_review_ts.to_dict()
H.gaps['first_review_ts'] = H.gaps['slug'].map(_new_first)
H.gaps['gap_days'] = (
    H.gaps['close_ts'] - H.gaps['first_review_ts']
).dt.total_seconds() / 86400
H.gap_lookup = dict(zip(H.gaps['slug'], H.gaps['gap_days']))
_activity = H.critic_activity_counts()
print(f'noon-shift: {_n_shifted} day-level reviews')


## Config


In [ ]:
SNAP_DAYS_LIST = [5, 4, 3, 2, 1]
PHASE2_CONST = 2.0          # reviews expected in 10h midnight-ET → 10am-ET close window
A1_POOL_SIZE = 20
TOP_TIER_N = 30
MIN_OBS_CRITICS = 3
ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
CV_FOLDS = 5
CV_SEED = 42

BASE_FEATURES = [
    'observed_count', 'first_review_dbc', 'target_gap', 'observed_rate',
    'rate_last_day', 'rate_first_day', 'top_critic_frac',
    'pub_diversity', 'pub_entropy', 'low_activity_frac',
]
TIER1_FEATURES = BASE_FEATURES + [
    'log_observed_count', 'log_rate_last_day', 'sqrt_rate_last_day', 'rate_delta',
]
POOL_FEATURES = ['remaining_base_rate_sum', 'pool_mass_consumed', 'observed_top_tier_frac']
ALL_FEATURES = TIER1_FEATURES + POOL_FEATURES

HM_TARGETS = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
              'they_will_kill_you', 'you_me_and_tuscany']

CACHE = H.CACHE_DIR / 'proposed_ship_stack_test.pkl'


## ET-convention snap and window helpers

For each target, midnight_et_close = midnight Eastern on close date (DST-aware via tz_convert). Phase-1 window = (midnight_et_close − N days, midnight_et_close]. Phase-2 window = (midnight_et_close, close_ts].


In [ ]:
def midnight_et_of_close(close_ts):
    """Return midnight Eastern on the close date (tz-aware, UTC-compatible)."""
    et = close_ts.tz_convert('US/Eastern')
    return et.normalize()  # 00:00 of close date, ET


def midnight_et_dbc_for(close_ts):
    """Days-before-close for midnight ET of close day. ~0.417 (=10h/24h) during EDT, ~0.458 during EST."""
    return (close_ts - midnight_et_of_close(close_ts)).total_seconds() / 86400


## A1 pool + base_rate lookup per target (LOO-clean)


In [ ]:
def build_a1_context(target_slug):
    close_ts = H.close_date_map[target_slug]
    training_slugs = H.default_training_slugs(
        H.movies, exclude_slug=target_slug,
        n=A1_POOL_SIZE, before_date=close_ts,
    )
    if len(training_slugs) < 5:
        return None
    train = H.reviews[H.reviews['movie_slug'].isin(training_slugs)]
    counts = train.groupby('reviewer_name')['movie_slug'].nunique()
    base_rate = (counts / A1_POOL_SIZE).to_dict()
    total_sum = float(sum(base_rate.values()))
    top_tier = set(counts.nlargest(TOP_TIER_N).index)
    return {'base_rate': base_rate, 'total_sum': total_sum, 'top_tier': top_tier}


a1_cache = {s: build_a1_context(s) for s in sorted(H.close_date_map)}
a1_cache = {s: c for s, c in a1_cache.items() if c is not None}
print(f'A1 pool built for {len(a1_cache)} targets')


## Feature + actual extraction for one (target, snap) under ET convention


In [ ]:
def extract_row(target_slug, snap_days):
    """Compute 17 features + actual phase1/phase2/total under ET convention. Or None if skip."""
    ctx = a1_cache.get(target_slug)
    if ctx is None:
        return None
    close_ts = H.close_date_map[target_slug]
    midnight_et_close = midnight_et_of_close(close_ts)
    midnight_et_dbc = (close_ts - midnight_et_close).total_seconds() / 86400

    snap_time = midnight_et_close - pd.Timedelta(days=snap_days)
    snap_dbc_eff = (close_ts - snap_time).total_seconds() / 86400

    state = H.snapshot_state(target_slug, snap_time)
    if state is None:
        return None
    if state['first_review_dbc'] < snap_dbc_eff + 1.0:
        return None
    if len(state['observed_critics']) < MIN_OBS_CRITICS:
        return None

    target_gap = H.gap_lookup.get(target_slug)
    if target_gap is None:
        return None
    obs_window_days = state['first_review_dbc'] - snap_dbc_eff
    if obs_window_days <= 0:
        return None

    # Observation-window feature stats
    mr = H.reviews[H.reviews['movie_slug'] == target_slug]
    obs = mr[(mr['estimated_timestamp'] < snap_time) & (mr['estimated_timestamp'] < close_ts)]
    first_rev_ts = obs['estimated_timestamp'].min()
    stats = H.observed_review_stats(target_slug, first_rev_ts, obs_window_days, _activity)

    last_day_start = snap_time - pd.Timedelta(days=1)
    rate_last_day = int(((obs['estimated_timestamp'] >= last_day_start)
                         & (obs['estimated_timestamp'] < snap_time)).sum())
    first_day_end = first_rev_ts + pd.Timedelta(days=1)
    rate_first_day = int(((obs['estimated_timestamp'] >= first_rev_ts)
                          & (obs['estimated_timestamp'] < first_day_end)).sum())

    feats = {
        'observed_count': state['observed_count'],
        'first_review_dbc': state['first_review_dbc'],
        'target_gap': target_gap,
        'observed_rate': state['observed_count'] / obs_window_days,
        'rate_last_day': rate_last_day,
        'rate_first_day': rate_first_day,
        'top_critic_frac': stats['top_critic_frac'],
        'pub_diversity': stats['pub_diversity'],
        'pub_entropy': stats['pub_entropy'],
        'low_activity_frac': stats['low_activity_frac'],
    }
    feats['log_observed_count'] = np.log1p(feats['observed_count'])
    feats['log_rate_last_day'] = np.log1p(feats['rate_last_day'])
    feats['sqrt_rate_last_day'] = np.sqrt(max(feats['rate_last_day'], 0))
    feats['rate_delta'] = feats['rate_last_day'] - feats['rate_first_day']

    # A1 pool features
    obs_set = state['observed_critics']
    obs_br_sum = float(sum(ctx['base_rate'].get(c, 0.0) for c in obs_set))
    total_sum = ctx['total_sum']
    remaining_br_sum = max(total_sum - obs_br_sum, 0.0)
    feats['remaining_base_rate_sum'] = remaining_br_sum
    feats['pool_mass_consumed'] = obs_br_sum / total_sum if total_sum > 0 else 0.0
    feats['observed_top_tier_frac'] = len(obs_set & ctx['top_tier']) / TOP_TIER_N

    # Actuals under ET convention
    # phase_1: reviews in (midnight_et_close − N days, midnight_et_close]
    # phase_2: reviews in (midnight_et_close, close_ts]
    # total:   reviews in (snap_time, close_ts]
    mr_dbc = ((close_ts - mr['estimated_timestamp']).dt.total_seconds() / 86400)
    phase1_mask = (mr_dbc > midnight_et_dbc) & (mr_dbc <= snap_dbc_eff)
    phase2_mask = (mr_dbc > 0) & (mr_dbc <= midnight_et_dbc)
    total_mask = (mr_dbc > 0) & (mr_dbc <= snap_dbc_eff)
    actual_phase1 = int(phase1_mask.sum())
    actual_phase2 = int(phase2_mask.sum())
    actual_total = int(total_mask.sum())

    return {
        'target_slug': target_slug,
        'snap_days': snap_days,
        'midnight_et_dbc': midnight_et_dbc,
        'snap_dbc_eff': snap_dbc_eff,
        'actual_phase1': actual_phase1,
        'actual_phase2': actual_phase2,
        'actual_total': actual_total,
        **feats,
    }


## Build feature matrix for cohort


In [ ]:
rows = []
start = time.time()
for slug in sorted(H.close_date_map):
    for snap in SNAP_DAYS_LIST:
        row = extract_row(slug, snap)
        if row is not None:
            rows.append(row)
df = pd.DataFrame(rows)
print(f'rows: {len(df)}   elapsed: {time.time()-start:.1f}s')
print(f'per-snap counts: {df["snap_days"].value_counts().sort_index().to_dict()}')
df.head()


## LOO Ridge fit per snap (target = actual_phase1)


In [ ]:
def select_alpha(X, y):
    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=CV_SEED)
    best_alpha, best_mae = None, np.inf
    for alpha in ALPHA_GRID:
        errs = []
        for tr_idx, te_idx in kf.split(X):
            pipe = Pipeline([('s', StandardScaler()), ('r', Ridge(alpha=alpha))])
            pipe.fit(X[tr_idx], y[tr_idx])
            errs.extend(np.abs(pipe.predict(X[te_idx]) - y[te_idx]).tolist())
        mae = float(np.mean(errs))
        if mae < best_mae:
            best_mae, best_alpha = mae, alpha
    return best_alpha


def loo_predict(X, y, alpha):
    preds = np.zeros(len(X))
    for i in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[i] = False
        pipe = Pipeline([('s', StandardScaler()), ('r', Ridge(alpha=alpha))])
        pipe.fit(X[mask], y[mask])
        preds[i] = pipe.predict(X[i:i+1])[0]
    return preds


snap_alpha = {}
df['phase1_pred'] = np.nan
for snap in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap]
    X = sub[ALL_FEATURES].values
    y = sub['actual_phase1'].values.astype(float)
    alpha = select_alpha(X, y)
    snap_alpha[snap] = alpha
    preds = loo_predict(X, y, alpha)
    df.loc[sub.index, 'phase1_pred'] = preds
    print(f'T-{snap}d (n={len(sub)}): α*={alpha}')

# Apply phase 2 constant + total
df['total_pred'] = df['phase1_pred'] + PHASE2_CONST
df['err_phase1'] = df['phase1_pred'] - df['actual_phase1']
df['err_total']  = df['total_pred'] - df['actual_total']

with open(CACHE, 'wb') as f:
    pickle.dump(df, f)
print(f'saved {CACHE}')


## Cohort-wide error summary (sanity check vs prior validation)


In [ ]:
print('=== cohort phase-1 (Ridge only, no phase-2) ===')
print(f'{"snap":<6}{"n":>5}{"MAE":>8}{"med|e|":>9}{"p90|e|":>9}{"me":>8}')
for snap in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap].dropna(subset=['phase1_pred'])
    err = sub['err_phase1']
    print(f'T-{snap}d{len(sub):>6}{err.abs().mean():>8.2f}{err.abs().median():>9.2f}'
          f'{err.abs().quantile(0.9):>9.2f}{err.mean():>+8.2f}')
print()
print('=== cohort total (Ridge + C=2) ===')
print(f'{"snap":<6}{"n":>5}{"MAE":>8}{"med|e|":>9}{"p90|e|":>9}{"me":>8}')
for snap in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap].dropna(subset=['total_pred'])
    err = sub['err_total']
    print(f'T-{snap}d{len(sub):>6}{err.abs().mean():>8.2f}{err.abs().median():>9.2f}'
          f'{err.abs().quantile(0.9):>9.2f}{err.mean():>+8.2f}')


## H/m per-movie results (the headline for this notebook)


In [ ]:
hm_df = df[df['target_slug'].isin(HM_TARGETS)].copy()

print('=== h/m per-target, per-snap: predicted_total vs actual_total ===\n')
print(f'{"target":<32}{"snap":>6}{"actual":>8}{"phase1":>8}{"phase2":>8}'
      f'{"pred_p1":>10}{"pred_tot":>10}{"err_tot":>10}')
for target in HM_TARGETS:
    for snap in SNAP_DAYS_LIST:
        row = hm_df[(hm_df['target_slug'] == target) & (hm_df['snap_days'] == snap)]
        if row.empty:
            print(f'{target:<32}  T-{snap}d   -- skipped --')
            continue
        r = row.iloc[0]
        print(f'{target:<32}{" T-"+str(snap)+"d":>6}{int(r["actual_total"]):>8}'
              f'{int(r["actual_phase1"]):>8}{int(r["actual_phase2"]):>8}'
              f'{r["phase1_pred"]:>10.2f}{r["total_pred"]:>10.2f}{r["err_total"]:>+10.2f}')
    print()


## H/m aggregate per snap


In [ ]:
print('=== h/m aggregate metrics (predicted_total vs actual_total) ===\n')
print(f'{"snap":<6}{"n":>4}{"MAE":>8}{"med|e|":>9}{"max|e|":>9}{"me":>8}  movies included')
for snap in SNAP_DAYS_LIST:
    sub = hm_df[hm_df['snap_days'] == snap]
    if sub.empty:
        continue
    err = sub['err_total']
    movies = ','.join(sorted(sub['target_slug']))
    print(f'T-{snap}d{len(sub):>5}{err.abs().mean():>8.2f}{err.abs().median():>9.2f}'
          f'{err.abs().max():>9.2f}{err.mean():>+8.2f}  {movies}')


## Sanity: phase-1 vs phase-2 decomposition on h/m

What was actually in phase 2? Is the C=2 assumption reasonable on the 10h ET window for these movies?


In [ ]:
print('=== phase-2 actuals (10h ET pre-market window) vs C=2 assumption ===\n')
print(f'{"target":<32}{"close_date_ET":<22}{"actual_phase2":>15}')
for target in HM_TARGETS:
    rows = hm_df[hm_df['target_slug'] == target]
    if rows.empty:
        continue
    close_et = H.close_date_map[target].tz_convert('US/Eastern')
    # phase_2 is the same regardless of snap (it's always the final 10h)
    actual_p2_values = rows['actual_phase2'].unique()
    val = actual_p2_values[0] if len(actual_p2_values) == 1 else str(sorted(actual_p2_values.tolist()))
    print(f'{target:<32}{str(close_et):<22}{val:>15}')

print(f'\nC = {PHASE2_CONST}  (applied to all targets as phase-2 constant)')


## Plot: predicted vs actual total per snap


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(SNAP_DAYS_LIST), figsize=(18, 4), sharey=False)
for ax, snap in zip(axes, SNAP_DAYS_LIST):
    sub_cohort = df[df['snap_days'] == snap].dropna(subset=['total_pred'])
    sub_hm = sub_cohort[sub_cohort['target_slug'].isin(HM_TARGETS)]
    ax.scatter(sub_cohort['actual_total'], sub_cohort['total_pred'],
               alpha=0.3, color='tab:gray', label=f'cohort (n={len(sub_cohort)})')
    ax.scatter(sub_hm['actual_total'], sub_hm['total_pred'],
               alpha=0.9, color='tab:red', s=60, label=f'h/m (n={len(sub_hm)})')
    lim = max(sub_cohort['actual_total'].max(), sub_cohort['total_pred'].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4)
    ax.set_title(f'T-{snap}d')
    ax.set_xlabel('actual total')
    ax.set_ylabel('predicted total')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Predicted total (Ridge + C=2) vs actual total — ET convention', y=1.05)
plt.tight_layout()
plt.show()


## Observations

*(fill in after run)*
